# Benchmark interactif

Ce notebook est destiné à l'analyse des résultats du benchmark. Il faut donc avoir lancé un run de benchmark au préalable (via python main.py) pour pouvoir analyser les résultats.
Ceux-ci proviennent de trois sources différentes :
- un fichier .csv nommé benchmark_results_{timestamp} (ou benchmark_results_latest_run) contenant les métriques
- un dossier logs contenant les logs d'entraînement des modèles de réseaux de neurones
- un dossier weights contenant les poids d'entraînement des modèles de réseaux de neurones

Ce notebook permet de :
1. Analyser les méthodes ayant obtenu les meilleurs résultats sur chaque dataset
2. Visualiser les clusters dans l'espace grâce à UMAP
3. Visualiser l'évolution des pertes d'entraînement grâce à Tensorboard

In [ ]:
import json
import glob
import umap
import torch
import logging
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from tensorboard import notebook
from scipy.optimize import linear_sum_assignment

from sklearn.exceptions import ConvergenceWarning
from numba.core.errors import NumbaPerformanceWarning

import pipelines.feature_based
import pipelines.autoencoder_based
import pipelines.series2vec_based

import features.handcrafted
import features.spectrogram
import features.rms

import models.mlp_ae
import models.conv1d_ae
import models.conv2d_ae
import models.series2vec_model

import clustering.algorithms

from config import build_config
from preprocessing.datasets import load_dataset, load_dataset_merged
from preprocessing.scalers import get_scaler
from pipelines.base import PIPELINE_REGISTRY
from clustering.base import CLUSTERING_REGISTRY
from models.base import MODEL_REGISTRY

warnings.simplefilter("ignore", category=UserWarning)
warnings.simplefilter("ignore", category=FutureWarning)
warnings.simplefilter("ignore", category=ConvergenceWarning)
warnings.simplefilter("ignore", category=NumbaPerformanceWarning)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

## 1. Panneau de configuration

In [ ]:
# Option 1 : "all" permet d'analyser les résultats de tous les datasets présents dans le .csv
# Option 2 : préciser à la main le nom d'un ou plusieurs datasets

DATASETS = ["all"]

run_dir = sorted(Path("results").glob("run__*"))[-1]
CSV_PATH = str(next(run_dir.glob("benchmark_results_*.csv")))
_, DATA_DIR, _, _ = build_config("config.yaml")

df_results = pd.read_csv(CSV_PATH)

if DATASETS == ["all"]:
  datasets = list(df_results["dataset"].unique())
else:
  datasets = [d for d in DATASETS if d in df_results["dataset"].unique()]

print(f"Fichier source : {CSV_PATH}")
print(f"Datasets à analyser : {datasets}")

## 2. Tableau des métriques

In [ ]:
pd.set_option("display.max_colwidth", None)

df_filtered = df_results[df_results["dataset"].isin(datasets)]

# Tableau 1
for dataset in datasets:
  print(f"Tableau des résultats détaillés pour {dataset}")
  df_ds = df_filtered[df_filtered["dataset"] == dataset].copy()

  df_ds_ranked = df_ds.sort_values(by=["ari", "accuracy"], ascending=False).reset_index(drop=True)

  cols = ["pipeline", "scaler", "clustering", "ari", "accuracy", "nmi", "silhouette", "davies_bouldin", "calinski_harabasz"]
  available_cols = [c for c in cols if c in df_ds_ranked.columns]
  display(df_ds_ranked[available_cols])

# Tableau 2
print("Tableau des meilleures performances pour chaque dataset")
summary_methods, summary_clust, summary_scale = [], [], []
for dataset in datasets:
  df_ds = df_filtered[df_filtered["dataset"] == dataset]

  best_per_method = df_ds.groupby("pipeline")["ari"].idxmax()
  df_best_methods = df_ds.loc[best_per_method]
  df_methods_ranked = df_best_methods.sort_values(by=["ari", "accuracy"], ascending=False).reset_index(drop=True)
  top_2_methods = df_methods_ranked.head(2)

  best_per_clust = df_ds.groupby("clustering")["ari"].idxmax()
  df_best_clust = df_ds.loc[best_per_clust]
  df_clust_ranked = df_best_clust.sort_values(by=["ari", "accuracy"], ascending=False).reset_index(drop=True)
  top_2_clust = df_clust_ranked.head(2)

  best_per_scale = df_ds.groupby("scaler")["ari"].idxmax()
  df_best_scale = df_ds.loc[best_per_scale]
  df_scale_ranked = df_best_scale.sort_values(by=["ari", "accuracy"], ascending=False).reset_index(drop=True)
  top_2_scale = df_scale_ranked.head(2)

  mean_ari = df_ds["ari"].mean()

  for rank, (_, row) in enumerate(top_2_methods.iterrows(), start=1):
    summary_methods.append({
        "Dataset": dataset,
        "Rang": f"#{rank}",
        "Méthode": row["pipeline"],
        "ARI": round(row["ari"], 3),
        "ARI moyen sur le dataset": round(mean_ari, 3),
        "Accuracy": round(row["accuracy"], 3)
    })

  for rank, (_, row) in enumerate(top_2_clust.iterrows(), start=1):
    summary_clust.append({
        "Dataset": dataset,
        "Rang": f"#{rank}",
        "Méthode": row["clustering"],
        "ARI": round(row["ari"], 3),
        "ARI moyen sur le dataset": round(mean_ari, 3),
        "Accuracy": round(row["accuracy"], 3)
    })

  for rank, (_, row) in enumerate(top_2_scale.iterrows(), start=1):
    summary_scale.append({
        "Dataset": dataset,
        "Rang": f"#{rank}",
        "Méthode": row["scaler"],
        "ARI": round(row["ari"], 3),
        "ARI moyen sur le dataset": round(mean_ari, 3),
        "Accuracy": round(row["accuracy"], 3)
    })

  df_summary_methods = pd.DataFrame(summary_methods)
  df_summary_clust = pd.DataFrame(summary_clust)
  df_summary_scale = pd.DataFrame(summary_scale)

  df_summary_methods = df_summary_methods.sort_values(by=["Dataset", "ARI"], ascending=[True, False]).reset_index(drop=True)
  df_summary_clust = df_summary_clust.sort_values(by=["Dataset", "ARI"], ascending=[True, False]).reset_index(drop=True)
  df_summary_scale = df_summary_scale.sort_values(by=["Dataset", "ARI"], ascending=[True, False]).reset_index(drop=True)

  display(df_summary_methods)
  display(df_summary_clust)
  display(df_summary_scale)

# Tableau 3
print("Tableau des meilleures performances globales par modèle")
summary_pipeline_methods = df_filtered.groupby("pipeline")[["ari", "accuracy", "nmi", "silhouette", "davies_bouldin", "calinski_harabasz"]].mean().sort_values(by="ari", ascending=False)
display(summary_pipeline_methods)

print("Tableau des meilleures performances globales par algo de clustering")
summary_pipeline_clust = df_filtered.groupby("clustering")[["ari", "accuracy", "nmi", "silhouette", "davies_bouldin", "calinski_harabasz"]].mean().sort_values(by="ari", ascending=False)
display(summary_pipeline_clust)

print("Tableau des meilleures performances globales par scaler")
summary_pipeline_scale = df_filtered.groupby("scaler")[["ari", "accuracy", "nmi", "silhouette", "davies_bouldin", "calinski_harabasz"]].mean().sort_values(by="ari", ascending=False)
display(summary_pipeline_scale)

## 3. Visualisation UMAP 3D

In [ ]:
all_configs, _, _, _ = build_config("config.yaml")
weights_dir = Path(CSV_PATH).parent / "weights"

for dataset in datasets:
  df_ds = df_results[df_results["dataset"] == dataset]
  df_ds = df_ds[df_ds["pipeline"] != "rms"]   # embedding constant

  with open(Path(DATA_DIR) / f"{dataset}_label_mapping.json", encoding="utf-8") as f:
    original_to_encoded = json.load(f)["original_to_encoded"]
  encoded_to_original = {int(v): k for k, v in original_to_encoded.items()}

  best_runs = [g.loc[g["ari"].idxmax()] for _, g in df_ds.groupby("pipeline", sort=False)]
  n_rows = len(best_runs)
  fig, axes = plt.subplots(n_rows, 2, figsize=(16, 8 * n_rows), subplot_kw={"projection": "3d"})
  axes = np.array(axes).reshape(n_rows, 2)

  for idx, row in enumerate(best_runs):
    method_name, scaler_name = row["pipeline"], row["scaler"]
    clust_name, run_name = row["clustering"], row["run_name"]
    print(f"Traitement de la config : {run_name}")

    cfg = next(c for c in all_configs
               if c.dataset_name == dataset and c.display_name == method_name
               and c.scaler_name == scaler_name and c.clustering_name == clust_name)

    # même split que le benchmark
    loader = load_dataset_merged if cfg.merge_train_test else load_dataset
    X_train, y_train, X_test, y_test = loader(dataset, DATA_DIR, test_size=cfg.merge_test_size, seed=cfg.seed)
    y_true = np.asarray(y_test)

    # embedding
    pipe = PIPELINE_REGISTRY[cfg.pipeline_name](**cfg.pipeline_params)
    weights_file = next((p for p in (weights_dir / f"weights__{run_name}.ckpt",
                                     weights_dir / f"weights__{run_name}.pth") if p.exists()), None)
    if weights_file is None:
      pipe.fit(X_train, y_train, X_test, y_test)
    elif method_name == "series2vec":
      pipe.load(weights_file, X_sample=X_test, num_classes=len(np.unique(y_train)))
    else:
      pipe.model = MODEL_REGISTRY[method_name](
        seq_len=X_train.shape[1], latent_dim=pipe.latent_dim, lr=pipe.lr, **pipe.model_params)
      ckpt = torch.load(weights_file, map_location="cpu", weights_only=False)
      pipe.model.load_state_dict(ckpt.get("state_dict", ckpt))
      pipe.model.eval()

    Z_train, Z_test = pipe.embed(X_train), pipe.embed(X_test)

    # scaling + clustering : même procédure que runner.py
    scaler = get_scaler(scaler_name)
    if scaler is not None:
      Z_train = scaler.fit_transform(Z_train)
      Z_test = scaler.transform(Z_test)

    n_clusters = cfg.n_clusters or len(set(y_train))
    clust = CLUSTERING_REGISTRY[clust_name](n_clusters=n_clusters, seed=cfg.seed, **cfg.clustering_params)
    clust.fit_predict(Z_train)
    try:
      y_pred = clust.predict(Z_test)
    except NotImplementedError:
      y_pred = clust.fit_predict(Z_test)

    emb_3d = umap.UMAP(n_components=3, random_state=42).fit_transform(Z_test)

    # alignement hongrois ; bruit et clusters non appariés = -999
    labels_true = np.unique(y_true)
    labels_pred = np.unique(y_pred[y_pred != -1])
    y_aligned = np.full(len(y_pred), -999)
    if len(labels_pred) > 0:
      cost = np.zeros((len(labels_pred), len(labels_true)), dtype=np.int64)
      for i, lp in enumerate(labels_pred):
        for j, lt in enumerate(labels_true):
          cost[i, j] = -np.sum((y_pred == lp) & (y_true == lt))
      r, c = linear_sum_assignment(cost)
      mapping = {labels_pred[a]: labels_true[b] for a, b in zip(r, c)}
      y_aligned = np.array([mapping.get(p, -999) for p in y_pred])

    ax1, ax2 = axes[idx, 0], axes[idx, 1]
    for class_id in labels_true:
      name = str(encoded_to_original.get(int(class_id), class_id))
      m = y_true == class_id
      ax1.scatter(emb_3d[m, 0], emb_3d[m, 1], emb_3d[m, 2], label=name, s=35, alpha=0.8)
      m = y_aligned == class_id
      if m.sum() > 0:
        ax2.scatter(emb_3d[m, 0], emb_3d[m, 1], emb_3d[m, 2], label=name, s=35, alpha=0.8)
    m = y_aligned == -999
    if m.sum() > 0:
      ax2.scatter(emb_3d[m, 0], emb_3d[m, 1], emb_3d[m, 2], color="gray", label="noise", s=15, alpha=0.3)

    ax1.set_title(f"Classes réelles\n[{run_name}]", fontsize=11)
    ax2.set_title(f"Clusters prédits\n[{run_name}]\nARI = {row['ari']:.2f} | Silhouette = {row['silhouette']:.2f}", fontsize=11)
    for ax in (ax1, ax2):
      ax.legend(loc="upper right", fontsize=7)
      ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])

  plt.tight_layout()
  plt.show()

## Tensorboard

In [ ]:
log_dirs = sorted(glob.glob("results/run_*/logs"))
latest_log = log_dirs[-1]
notebook.start(f"--logdir {Path(CSV_PATH).parent / 'logs'}")